# Taller Corte 2 — Sistema de Logística Escalado
## Grupo 5: Sebastián Vanoy, Samuel Lozano, Samuel Gonzales

**Reto escalado:** Reto 1 — Logística de paquetes y envíos

**Arquitectura:** POO con encapsulamiento + SQLite con relaciones PK/FK + CRUD

### Estructura del sistema

**Clases:**
- Persona (padre) -> Cliente (hijo)
- Envio (padre) -> Paquete (hijo)
- Ruta (padre) -> Destino (hijo)

**Tablas:**
- clientes (id_cliente PK, nombre, telefono, email)
- destinos (id_destino PK, ciudad, tarifa_base)
- paquetes (id_paquete PK, peso, tipo, costo, id_cliente FK, id_destino FK)

## Celda 1 — Importaciones y Conexión a la Base de Datos

In [7]:
import sqlite3

# Conexión a archivo persistente .db
try:
    conn = sqlite3.connect("base_datos_grupo_5.db")
    conn.row_factory = sqlite3.Row          # permite acceder columnas por nombre
    conn.execute("PRAGMA foreign_keys = ON")  # activa integridad referencial
    cursor = conn.cursor()
    print("✅ Conexión exitosa a base_datos_grupo_5.db")
except sqlite3.Error as e:
    print(f"❌ Error al conectar con la base de datos: {e}")

✅ Conexión exitosa a base_datos_grupo_5.db


## Celda 2 — Creación de Tablas (PK y FK)

In [8]:
try:
    # Tabla clientes
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS clientes (
            id_cliente INTEGER PRIMARY KEY AUTOINCREMENT,
            nombre     TEXT    NOT NULL,
            telefono   TEXT    NOT NULL,
            email      TEXT    NOT NULL
        )
    """)

    # Tabla destinos
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS destinos (
            id_destino  INTEGER PRIMARY KEY AUTOINCREMENT,
            ciudad      TEXT    NOT NULL,
            tarifa_base REAL    NOT NULL
        )
    """)

    # Tabla paquetes — referencia clientes y destinos mediante FK
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS paquetes (
            id_paquete INTEGER PRIMARY KEY AUTOINCREMENT,
            peso       REAL    NOT NULL,
            tipo       TEXT    NOT NULL,
            costo      REAL    NOT NULL,
            id_cliente INTEGER NOT NULL,
            id_destino INTEGER NOT NULL,
            FOREIGN KEY (id_cliente) REFERENCES clientes(id_cliente),
            FOREIGN KEY (id_destino) REFERENCES destinos(id_destino)
        )
    """)

    conn.commit()
    print("✅ Tablas creadas correctamente (clientes, destinos, paquetes)")
except sqlite3.Error as e:
    print(f"❌ Error al crear tablas: {e}")

✅ Tablas creadas correctamente (clientes, destinos, paquetes)


## Celda 3 — Definición de Clases (POO + Encapsulamiento)

In [9]:
# ─────────────────────────────────────────────
# CLASE PADRE 1: Persona
# ─────────────────────────────────────────────
class Persona:
    """
    Clase padre que representa a cualquier persona en el sistema.
    Atributos privados con acceso mediante get/set.
    """
    def __init__(self, nombre, telefono):
        self.__nombre   = nombre
        self.__telefono = telefono

    # --- Getters ---
    def get_nombre(self):
        return self.__nombre

    def get_telefono(self):
        return self.__telefono

    # --- Setters con validación ---
    def set_nombre(self, nombre):
        if not nombre.strip():
            raise ValueError("El nombre no puede estar vacío.")
        self.__nombre = nombre

    def set_telefono(self, telefono):
        if not telefono.strip():
            raise ValueError("El teléfono no puede estar vacío.")
        self.__telefono = telefono

    def __str__(self):
        return f"Persona: {self.__nombre} | Tel: {self.__telefono}"


# ─────────────────────────────────────────────
# CLASE HIJO 1: Cliente  (hija de Persona)
# ─────────────────────────────────────────────
class Cliente(Persona):
    """
    Extiende Persona añadiendo email.
    Incluye métodos para guardar y consultar en la BD.
    """
    def __init__(self, nombre, telefono, email):
        super().__init__(nombre, telefono)  # reutiliza __init__ del padre
        self.__email = email

    def get_email(self):
        return self.__email

    def set_email(self, email):
        if "@" not in email:
            raise ValueError("El email debe contener '@'.")
        self.__email = email

    def guardar(self, conn):
        """Inserta el cliente en la tabla clientes."""
        try:
            conn.execute(
                "INSERT INTO clientes (nombre, telefono, email) VALUES (?, ?, ?)",
                (self.get_nombre(), self.get_telefono(), self.__email)
            )
            conn.commit()
        except sqlite3.Error as e:
            raise RuntimeError(f"Error al guardar cliente: {e}")

    @staticmethod
    def listar(conn):
        """Retorna todos los clientes de la BD."""
        try:
            return conn.execute("SELECT * FROM clientes").fetchall()
        except sqlite3.Error as e:
            raise RuntimeError(f"Error al listar clientes: {e}")

    def __str__(self):
        return (f"Cliente: {self.get_nombre()} | "
                f"Tel: {self.get_telefono()} | Email: {self.__email}")


# ─────────────────────────────────────────────
# CLASE PADRE 2: Ruta
# ─────────────────────────────────────────────
class Ruta:
    """
    Clase padre que representa una ruta genérica de transporte.
    """
    def __init__(self, ciudad, tarifa_base):
        self.__ciudad      = ciudad
        self.__tarifa_base = tarifa_base

    def get_ciudad(self):
        return self.__ciudad

    def get_tarifa_base(self):
        return self.__tarifa_base

    def set_ciudad(self, ciudad):
        if not ciudad.strip():
            raise ValueError("La ciudad no puede estar vacía.")
        self.__ciudad = ciudad

    def set_tarifa_base(self, tarifa):
        if tarifa <= 0:
            raise ValueError("La tarifa debe ser mayor a 0.")
        self.__tarifa_base = tarifa

    def __str__(self):
        return f"Ruta: {self.__ciudad} | Tarifa base: ${self.__tarifa_base:,.0f}"


# ─────────────────────────────────────────────
# CLASE HIJO 2: Destino  (hija de Ruta)
# ─────────────────────────────────────────────
class Destino(Ruta):
    """
    Extiende Ruta con métodos de persistencia en la BD.
    """
    def __init__(self, ciudad, tarifa_base):
        super().__init__(ciudad, tarifa_base)

    def guardar(self, conn):
        """Inserta el destino en la tabla destinos."""
        try:
            conn.execute(
                "INSERT INTO destinos (ciudad, tarifa_base) VALUES (?, ?)",
                (self.get_ciudad(), self.get_tarifa_base())
            )
            conn.commit()
        except sqlite3.Error as e:
            raise RuntimeError(f"Error al guardar destino: {e}")

    @staticmethod
    def listar(conn):
        """Retorna todos los destinos."""
        try:
            return conn.execute("SELECT * FROM destinos").fetchall()
        except sqlite3.Error as e:
            raise RuntimeError(f"Error al listar destinos: {e}")

    def __str__(self):
        return f"Destino → {self.get_ciudad()} | Tarifa: ${self.get_tarifa_base():,.0f}/kg"


# ─────────────────────────────────────────────
# CLASE PADRE 3: Envio
# ─────────────────────────────────────────────
class Envio:
    """
    Clase padre que representa un envío genérico.
    Incluye la lógica de clasificación por peso.
    """
    def __init__(self, peso, id_cliente, id_destino):
        self.__peso       = peso
        self.__id_cliente = id_cliente
        self.__id_destino = id_destino

    def get_peso(self):
        return self.__peso

    def get_id_cliente(self):
        return self.__id_cliente

    def get_id_destino(self):
        return self.__id_destino

    def set_peso(self, peso):
        if peso <= 0:
            raise ValueError("El peso debe ser mayor a 0.")
        self.__peso = peso

    def set_id_destino(self, id_destino):
        if id_destino <= 0:
            raise ValueError("ID de destino inválido.")
        self.__id_destino = id_destino

    def clasificar(self):
        """Clasifica el paquete según su peso."""
        if self.__peso < 1:
            return "Documento"
        elif self.__peso < 10:
            return "Paquetería"
        else:
            return "Carga"

    def __str__(self):
        return (f"Envio: {self.__peso} kg | "
                f"Tipo: {self.clasificar()} | "
                f"Cliente ID: {self.__id_cliente} | "
                f"Destino ID: {self.__id_destino}")


# ─────────────────────────────────────────────
# CLASE HIJO 3: Paquete  (hija de Envio)
# ─────────────────────────────────────────────
class Paquete(Envio):
    """
    Extiende Envio. Calcula el costo automáticamente
    según la tarifa del destino y ofrece CRUD contra la BD.
    """
    def __init__(self, peso, id_cliente, id_destino, tarifa):
        super().__init__(peso, id_cliente, id_destino)
        self.__tarifa = tarifa
        self.__costo  = round(self.get_peso() * tarifa, 2)

    def get_costo(self):
        return self.__costo

    def recalcular_costo(self, tarifa):
        """Recalcula el costo si cambia el peso o el destino."""
        self.__tarifa = tarifa
        self.__costo  = round(self.get_peso() * tarifa, 2)

    def guardar(self, conn):
        """INSERT: inserta el paquete en la BD."""
        try:
            conn.execute(
                """
                INSERT INTO paquetes (peso, tipo, costo, id_cliente, id_destino)
                VALUES (?, ?, ?, ?, ?)
                """,
                (self.get_peso(), self.clasificar(), self.__costo,
                 self.get_id_cliente(), self.get_id_destino())
            )
            conn.commit()
        except sqlite3.Error as e:
            raise RuntimeError(f"Error al guardar paquete: {e}")

    @staticmethod
    def listar_con_join(conn):
        """
        SELECT con JOIN: une paquetes, clientes y destinos.
        Retorna el manifiesto completo.
        """
        try:
            return conn.execute("""
                SELECT
                    p.id_paquete,
                    c.nombre      AS cliente,
                    d.ciudad      AS destino,
                    p.peso,
                    p.tipo,
                    p.costo
                FROM paquetes p
                JOIN clientes c ON p.id_cliente = c.id_cliente
                JOIN destinos d ON p.id_destino = d.id_destino
                ORDER BY p.peso ASC
            """).fetchall()
        except sqlite3.Error as e:
            raise RuntimeError(f"Error al listar paquetes: {e}")

    @staticmethod
    def actualizar(conn, id_paquete, nuevo_peso, nuevo_id_destino):
        """
        UPDATE: actualiza peso, tipo, costo e id_destino de un paquete.
        Recalcula tipo y costo consultando la tarifa del nuevo destino.
        """
        try:
            fila = conn.execute(
                "SELECT tarifa_base FROM destinos WHERE id_destino = ?",
                (nuevo_id_destino,)
            ).fetchone()
            if not fila:
                raise ValueError(f"No existe destino con ID {nuevo_id_destino}.")

            tarifa = fila["tarifa_base"]
            nuevo_costo = round(nuevo_peso * tarifa, 2)

            # Reclasificar
            if nuevo_peso < 1:
                nuevo_tipo = "Documento"
            elif nuevo_peso < 10:
                nuevo_tipo = "Paquetería"
            else:
                nuevo_tipo = "Carga"

            conn.execute("""
                UPDATE paquetes
                SET peso = ?, tipo = ?, costo = ?, id_destino = ?
                WHERE id_paquete = ?
            """, (nuevo_peso, nuevo_tipo, nuevo_costo, nuevo_id_destino, id_paquete))
            conn.commit()
        except sqlite3.Error as e:
            raise RuntimeError(f"Error al actualizar paquete: {e}")

    @staticmethod
    def eliminar(conn, id_paquete):
        """DELETE: elimina un paquete por su ID."""
        try:
            conn.execute(
                "DELETE FROM paquetes WHERE id_paquete = ?",
                (id_paquete,)
            )
            conn.commit()
        except sqlite3.Error as e:
            raise RuntimeError(f"Error al eliminar paquete: {e}")


print("Clases definidas: Persona → Cliente | Ruta → Destino | Envio → Paquete")

Clases definidas: Persona → Cliente | Ruta → Destino | Envio → Paquete


## Celda 4 — Datos Iniciales (mínimo 5 registros por tabla)

In [10]:
# Solo inserta si las tablas están vacías (evita duplicados al re-ejecutar)
try:
    if conn.execute("SELECT COUNT(*) FROM clientes").fetchone()[0] == 0:

        # ── 5 Clientes ──────────────────────────────────────────────────────────
        clientes_iniciales = [
            Cliente("Carlos Mendez",  "3101234567", "carlos@email.com"),
            Cliente("Laura Ríos",     "3209876543", "laura@email.com"),
            Cliente("Pedro Alonso",   "3154561234", "pedro@email.com"),
            Cliente("Valentina Cruz", "3007891234", "valen@email.com"),
            Cliente("Andrés Torres",  "3123456789", "andres@email.com"),
        ]
        for c in clientes_iniciales:
            c.guardar(conn)

        # ── 5 Destinos ──────────────────────────────────────────────────────────
        destinos_iniciales = [
            Destino("Bogotá",   5000),
            Destino("Medellín", 6000),
            Destino("Yopal",    7000),
            Destino("Garagoa",  4500),
            Destino("Duitama",  5500),
        ]
        for d in destinos_iniciales:
            d.guardar(conn)

        # ── 5 Paquetes ──────────────────────────────────────────────────────────
        # Paquete(peso, id_cliente, id_destino, tarifa)
        paquetes_iniciales = [
            Paquete(0.5,  1, 4, 4500),   # Documento → Garagoa
            Paquete(9.0,  2, 3, 7000),   # Paquetería → Yopal
            Paquete(23.0, 3, 4, 4500),   # Carga → Garagoa
            Paquete(30.0, 4, 1, 5000),   # Carga → Bogotá
            Paquete(34.0, 5, 5, 5500),   # Carga → Duitama
        ]
        for p in paquetes_iniciales:
            p.guardar(conn)

        print("Datos iniciales insertados: 5 clientes, 5 destinos, 5 paquetes")
    else:
        print("ℹ️  Los datos iniciales ya existen en la base de datos.")

except (sqlite3.Error, RuntimeError, ValueError) as e:
    print(f"Error al insertar datos iniciales: {e}")

ℹ️  Los datos iniciales ya existen en la base de datos.


## Celda 5 — Funciones Auxiliares del Menú

In [11]:
def mostrar_clientes():
    """Imprime todos los clientes registrados."""
    filas = Cliente.listar(conn)
    if not filas:
        print("No hay clientes registrados.")
        return
    print(f"\n{'ID':<5} {'Nombre':<20} {'Teléfono':<15} {'Email'}")
    print("-" * 60)
    for f in filas:
        print(f"{f['id_cliente']:<5} {f['nombre']:<20} {f['telefono']:<15} {f['email']}")


def mostrar_destinos():
    """Imprime todos los destinos disponibles."""
    filas = Destino.listar(conn)
    if not filas:
        print("No hay destinos registrados.")
        return
    print(f"\n{'ID':<5} {'Ciudad':<15} {'Tarifa base ($/kg)'}")
    print("-" * 40)
    for f in filas:
        print(f"{f['id_destino']:<5} {f['ciudad']:<15} ${f['tarifa_base']:,.0f}")


def mostrar_manifiesto():
    """Imprime el manifiesto completo usando JOIN."""
    filas = Paquete.listar_con_join(conn)
    if not filas:
        print("No hay paquetes registrados.")
        return
    print(f"\n{'ID':<5} {'Cliente':<20} {'Destino':<12} {'Peso (kg)':<12} {'Tipo':<12} {'Costo'}")
    print("-" * 75)
    for f in filas:
        print(f"{f['id_paquete']:<5} {f['cliente']:<20} {f['destino']:<12} "
              f"{f['peso']:<12} {f['tipo']:<12} ${f['costo']:,.2f}")


print("Funciones auxiliares definidas")

Funciones auxiliares definidas


## Celda 6 — Menú Interactivo Principal (CRUD completo)

In [ ]:
while True:
    print("\n" + "═" * 45)
    print("   SISTEMA DE LOGÍSTICA — Logística de Paquetes")
    print("═" * 45)
    print("  1. Registrar nuevo cliente")
    print("  2. Registrar nuevo destino")
    print("  3. Registrar nuevo paquete  [CREATE]")
    print("  4. Ver manifiesto (JOIN)    [READ]")
    print("  5. Ver clientes")
    print("  6. Ver destinos")
    print("  7. Actualizar paquete       [UPDATE]")
    print("  8. Eliminar paquete         [DELETE]")
    print("  9. Salir")
    print("═" * 45)

    opcion = input("Opción: ").strip()

    # ── 1. REGISTRAR CLIENTE ────────────────────────────────────────────────
    if opcion == "1":
        try:
            nombre   = input("Nombre del cliente: ").strip()
            telefono = input("Teléfono: ").strip()
            email    = input("Email: ").strip()
            c = Cliente(nombre, telefono, email)  # valida en __init__/set
            c.guardar(conn)
            print(f"Cliente '{nombre}' registrado correctamente.")
        except (ValueError, RuntimeError) as e:
            print(f"Error: {e}")

    # ── 2. REGISTRAR DESTINO ────────────────────────────────────────────────
    elif opcion == "2":
        try:
            ciudad = input("Ciudad destino: ").strip()
            tarifa = float(input("Tarifa base ($/kg): "))
            d = Destino(ciudad, tarifa)
            d.guardar(conn)
            print(f"Destino '{ciudad}' registrado con tarifa ${tarifa:,.0f}/kg.")
        except ValueError as e:
            print(f"Valor inválido: {e}")
        except RuntimeError as e:
            print(f"Error de BD: {e}")

    # ── 3. REGISTRAR PAQUETE (CREATE) ───────────────────────────────────────
    elif opcion == "3":
        try:
            mostrar_clientes()
            id_cliente = int(input("\nID del cliente: "))
            mostrar_destinos()
            id_destino = int(input("\nID del destino: "))
            peso = float(input("Peso del paquete (kg): "))

            # Obtener tarifa del destino elegido
            fila = conn.execute(
                "SELECT tarifa_base FROM destinos WHERE id_destino = ?",
                (id_destino,)
            ).fetchone()
            if not fila:
                raise ValueError(f"No existe destino con ID {id_destino}.")

            p = Paquete(peso, id_cliente, id_destino, fila["tarifa_base"])
            p.guardar(conn)
            print(f"Paquete registrado: {p.clasificar()} | "
                  f"{peso} kg | Costo: ${p.get_costo():,.2f}")
        except ValueError as e:
            print(f"Valor inválido: {e}")
        except RuntimeError as e:
            print(f"Error de BD: {e}")

    # ── 4. VER MANIFIESTO con JOIN (READ) ───────────────────────────────────
    elif opcion == "4":
        try:
            mostrar_manifiesto()
        except RuntimeError as e:
            print(f"Error: {e}")

    # ── 5. VER CLIENTES ─────────────────────────────────────────────────────
    elif opcion == "5":
        try:
            mostrar_clientes()
        except RuntimeError as e:
            print(f"Error: {e}")

    # ── 6. VER DESTINOS ─────────────────────────────────────────────────────
    elif opcion == "6":
        try:
            mostrar_destinos()
        except RuntimeError as e:
            print(f"Error: {e}")

    # ── 7. ACTUALIZAR PAQUETE (UPDATE) ──────────────────────────────────────
    elif opcion == "7":
        try:
            mostrar_manifiesto()
            id_paquete     = int(input("\nID del paquete a actualizar: "))
            nuevo_peso     = float(input("Nuevo peso (kg): "))
            mostrar_destinos()
            nuevo_destino  = int(input("\nNuevo ID de destino: "))

            Paquete.actualizar(conn, id_paquete, nuevo_peso, nuevo_destino)
            print(f"Paquete {id_paquete} actualizado correctamente.")
        except ValueError as e:
            print(f"Valor inválido: {e}")
        except RuntimeError as e:
            print(f"Error de BD: {e}")

    # ── 8. ELIMINAR PAQUETE (DELETE) ────────────────────────────────────────
    elif opcion == "8":
        try:
            mostrar_manifiesto()
            id_paquete = int(input("\nID del paquete a eliminar: "))
            confirmar  = input(f"¿Confirmar eliminación del paquete {id_paquete}? (si/no): ").strip().lower()
            if confirmar == "si":
                Paquete.eliminar(conn, id_paquete)
                print(f"Paquete {id_paquete} eliminado.")
            else:
                print("Operación cancelada.")
        except ValueError as e:
            print(f"Valor inválido: {e}")
        except RuntimeError as e:
            print(f"Error de BD: {e}")

    # ── 9. SALIR ────────────────────────────────────────────────────────────
    elif opcion == "9":
        try:
            conn.close()
        except sqlite3.Error:
            pass
        print("👋 Conexión cerrada. ¡Hasta luego!")
        break

    else:
        print("⚠️  Opción inválida. Elige entre 1 y 9.")


═════════════════════════════════════════════
   SISTEMA DE LOGÍSTICA — Logística de Paquetes
═════════════════════════════════════════════
  1. Registrar nuevo cliente
  2. Registrar nuevo destino
  3. Registrar nuevo paquete  [CREATE]
  4. Ver manifiesto (JOIN)    [READ]
  5. Ver clientes
  6. Ver destinos
  7. Actualizar paquete       [UPDATE]
  8. Eliminar paquete         [DELETE]
  9. Salir
═════════════════════════════════════════════

ID    Cliente              Destino      Peso (kg)    Tipo         Costo
---------------------------------------------------------------------------
1     Carlos Mendez        Garagoa      0.5          Documento    $2,250.00
2     Laura Ríos           Yopal        9.0          Paquetería   $63,000.00
3     Pedro Alonso         Garagoa      23.0         Carga        $103,500.00
4     Valentina Cruz       Bogotá       30.0         Carga        $150,000.00
5     Andrés Torres        Duitama      34.0         Carga        $187,000.00

═════════════════